In [1]:
%pip install -q pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
pd.set_option("display.max_columns",None)

In [6]:
train_df=pd.read_csv("titanic_train_imputed.csv")
test_df=pd.read_csv("titanic_test_raw.csv")
target="survived"
X_train=train_df.drop(columns=target)
y_train=train_df[target]

In [9]:
X_train.head()

,Unnamed: 0,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,0,3,male,24.0,2,0,24.1500,S,Third,man,True,C,Southampton,False
1,1,3,male,44.0,0,1,16.1000,S,Third,man,True,C,Southampton,False
2,2,3,male,22.0,0,0,7.2250,C,Third,man,True,C,Cherbourg,True
3,3,3,male,41.0,2,0,14.1083,S,Third,man,True,C,Southampton,False
4,4,3,female,28.5,1,0,15.5000,Q,Third,woman,False,C,Queenstown,False


In [8]:
y_train.shape

(179,)

In [10]:

X_test=test_df.drop(columns=target)
y_test=test_df[target]

In [11]:
X_test.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,24.0,2,0,24.1500,S,Third,man,True,NaN,Southampton,False
1,3,male,44.0,0,1,16.1000,S,Third,man,True,NaN,Southampton,False
2,3,male,22.0,0,0,7.2250,C,Third,man,True,NaN,Cherbourg,True
3,3,male,41.0,2,0,14.1083,S,Third,man,True,NaN,Southampton,False
4,3,female,NaN,1,0,15.5000,Q,Third,woman,False,NaN,Queenstown,False


In [12]:
X_train=X_train.drop(columns=["class","embark_town"])
X_test=X_test.drop(columns=["class","embark_town"])


In [13]:
X_train.head()

,Unnamed: 0,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,deck,alone
0,0,3,male,24.0,2,0,24.1500,S,man,True,C,False
1,1,3,male,44.0,0,1,16.1000,S,man,True,C,False
2,2,3,male,22.0,0,0,7.2250,C,man,True,C,True
3,3,3,male,41.0,2,0,14.1083,S,man,True,C,False
4,4,3,female,28.5,1,0,15.5000,Q,woman,False,C,False


In [14]:
X_train.columns.to_list()

['Unnamed: 0',
 'pclass',
 'sex',
 'age',
 'sibsp',
 'parch',
 'fare',
 'embarked',
 'who',
 'adult_male',
 'deck',
 'alone']

In [15]:
nums_cols=["age","sibsp","parch","fare"]
cat_cols=['pclass',
 'sex',
 'embarked',
 'who',
 'adult_male',
 'deck',
 'alone']

split into binary, ordinal categories (lable Encoding ) and nomial encoding (One hot Encoding)

In [19]:
for col in cat_cols:
    print(col,X_train[col].unique())
    print("-"*50)

pclass [3 2 1]
--------------------------------------------------
sex <ArrowStringArray>
['male', 'female']
Length: 2, dtype: str
--------------------------------------------------
embarked <ArrowStringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
--------------------------------------------------
who <ArrowStringArray>
['man', 'woman', 'child']
Length: 3, dtype: str
--------------------------------------------------
adult_male [ True False]
--------------------------------------------------
deck <ArrowStringArray>
['C', 'B', 'F', 'E', 'D', 'A']
Length: 6, dtype: str
--------------------------------------------------
alone [False  True]
--------------------------------------------------


In [20]:
binary_ordinal_col=['sex',"adult_male","alone"]#label encoding
nominal_col=["embarked","who","deck"]# onehot Encoding

In [29]:
X_train_enc=X_train.copy()
X_test_enc=X_test.copy()


**lable Enocding**

In [30]:
label_encoders={}
for col in binary_ordinal_col:
    le=LabelEncoder()
    le.fit(X_train[col])
    label_encoders[col]=le
    X_train_enc[col]=le.transform(X_train[col])
    X_test_enc[col]=le.transform(X_test[col])
    
    
    
      

In [31]:
X_train_enc.head()

,Unnamed: 0,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,deck,alone
0,0,3,1,24.0,2,0,24.1500,S,man,1,C,0
1,1,3,1,44.0,0,1,16.1000,S,man,1,C,0
2,2,3,1,22.0,0,0,7.2250,C,man,1,C,1
3,3,3,1,41.0,2,0,14.1083,S,man,1,C,0
4,4,3,0,28.5,1,0,15.5000,Q,woman,0,C,0


One Hot Encoding

dummy variable trap

if there are n categories we  need only n-1 columns

In [33]:
ohe=OneHotEncoder(drop="first",handle_unknown="ignore",sparse_output=False )


In [34]:
ohe.fit(X_train[nominal_col])

,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infre

In [36]:
train_ohe=ohe.transform(X_train[nominal_col])
test_ohe=ohe.transform(X_test[nominal_col])


c:\Users\kaush\Desktop\SummerBreak\Machine Learning\MachineLearning\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [39]:
ohe_cols=ohe.get_feature_names_out(nominal_col)
ohe_cols

array(['embarked_Q', 'embarked_S', 'who_man', 'who_woman', 'deck_B',
       'deck_C', 'deck_D', 'deck_E', 'deck_F'], dtype=object)

In [42]:
train_ohe

array([[0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.]], shape=(179, 9))

In [60]:
train_ohe_df=pd.DataFrame(train_ohe,columns=ohe_cols,index=X_train.index)
test_ohe_df=pd.DataFrame(test_ohe,columns=ohe_cols,index=X_test.index)


In [61]:
train_ohe_df.shape

(179, 9)

In [45]:
X_train_enc.head()

,Unnamed: 0,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,deck,alone
0,0,3,1,24.0,2,0,24.1500,S,man,1,C,0
1,1,3,1,44.0,0,1,16.1000,S,man,1,C,0
2,2,3,1,22.0,0,0,7.2250,C,man,1,C,1
3,3,3,1,41.0,2,0,14.1083,S,man,1,C,0
4,4,3,0,28.5,1,0,15.5000,Q,woman,0,C,0


In [47]:
X_train_enc=X_train_enc.drop(columns=nominal_col)
X_test_enc=X_test_enc.drop(columns=nominal_col)



In [48]:
X_train_enc.head()

,Unnamed: 0,pclass,sex,age,sibsp,parch,fare,adult_male,alone
0,0,3,1,24.0,2,0,24.1500,1,0
1,1,3,1,44.0,0,1,16.1000,1,0
2,2,3,1,22.0,0,0,7.2250,1,1
3,3,3,1,41.0,2,0,14.1083,1,0
4,4,3,0,28.5,1,0,15.5000,0,0


In [53]:
X_train_enc=pd.concat([X_train_enc,train_ohe_df],axis=1)
X_test_enc=pd.concat([X_test_enc,test_ohe_df],axis=1)


In [50]:
print(train_ohe_df.shape)

(179, 9)


In [52]:
print(X_train_enc.shape)

(179, 18)


In [54]:
X_train_enc.dtypes

Unnamed: 0      int64
pclass          int64
sex             int64
age           float64
sibsp           int64
parch           int64
fare          float64
adult_male      int64
alone           int64
embarked_Q    float64
embarked_S    float64
who_man       float64
who_woman     float64
deck_B        float64
deck_C        float64
deck_D        float64
deck_E        float64
deck_F        float64
embarked_Q    float64
embarked_S    float64
who_man       float64
who_woman     float64
deck_B        float64
deck_C        float64
deck_D        float64
deck_E        float64
deck_F        float64
dtype: object

In [56]:
train_enc_df=X_train_enc.copy()
train_enc_df[target]=y_train.values

test_enc_df=X_test_enc.copy()
test_enc_df[target]=y_test.values

In [59]:
train_enc_df.shape

(179, 28)

In [58]:
train_enc_df.to_csv("titanic_train_encoded.csv",index=False)
test_enc_df.to_csv("titanic_test_encoded.csv",index=False)
